# Custom Strategies

**Audience:** method authors who implement a selection, estimation or stopping method that catsim
lacks. **Level:** advanced. You know item response theory and Python classes, and have run a
simulation, as the [getting-started notebook](01_getting_started.ipynb) does.

In this notebook you will:

- write a selector and a stopper by subclassing catsim's base classes;
- read what a strategy gets from its step context and from an estimate;
- call a strategy directly on a session, with `engine.context(session)`;
- run each strategy in a live session through `CatEngine` and in a simulation through
  `SimulationRunner`, like the built-in ones.

It is the hands-on companion of the [Extending catsim](https://douglasrizzo.com.br/catsim/extending.html) guide, which lists what each
strategy receives and returns, the rules every strategy keeps and the helpers selectors share. To
run it outside this repository, install catsim first: `pip install catsim`.

In [ ]:
import dataclasses
from dataclasses import dataclass

import numpy
from typing_extensions import override

from catsim import CatEngine, ItemBank, RunContext, SimulatedResponseProvider, SimulationRunner, StepContext
from catsim.estimation import BaseEstimator, EAPEstimator
from catsim.initialization import FixedPointInitializer
from catsim.selection import BaseSelector, MaxInfoSelector, best_pool, pick_best, remaining
from catsim.stopping import BaseStopper, MaxLengthStopper, StopRule

## A Bank and a Population

A generated bank of 300 items under the three-parameter logistic model, and 200 simulated
examinees whose abilities are evenly spread from -2.5 to 2.5, so that every part of the scale has
examinees. Every engine here starts at an ability estimate of 0 and estimates ability by expected
a posteriori (EAP); `engine_with` builds one from a selector and a stopper. `BANK_SEED` draws the
bank, and `RUN_SEED` seeds each run's generator, the one source of randomness for the strategies
and the simulated responses.

In [ ]:
BANK_SEED = 11
RUN_SEED = 5
N_ITEMS = 300
N_EXAMINEES = 200

item_bank = ItemBank.generate(N_ITEMS, "3PL", rng=BANK_SEED)
true_thetas = numpy.linspace(-2.5, 2.5, N_EXAMINEES)
estimator = EAPEstimator()


def engine_with(selector: BaseSelector, stopper: BaseStopper) -> CatEngine:
  """Build an engine that starts at 0 and estimates by EAP, with the given selector and stopper."""
  return CatEngine(item_bank, FixedPointInitializer(0.0), selector, estimator, stopper)

## A Custom Selector

Under maximum-information selection, every examinee gets the same first item, since all start at
the same estimate, and that item's exposure rate is 1. The first items are chosen with a rough
estimate anyway, so the item most informative at the estimate may not be the most informative at
the examinee's ability. `EarlyRandomesqueSelector` draws at random among the `pool_size` most
informative items while the standard error is above `until_se`, and gives the most informative
item after that.

A selector subclasses `BaseSelector` and overrides `select`, which receives a `StepContext` and
returns an item index. This one reads the item bank, the estimate `theta` and its standard error
`se` from the context; `remaining`, `best_pool` and `pick_best` are the helpers the built-in
selectors use ([Extending catsim](https://douglasrizzo.com.br/catsim/extending.html#the-selector-helpers)). Its random draw comes from
`context.rng`, the run's generator. `@override`, from `typing_extensions`, lets a type checker
report a method that overrides nothing. The class is a frozen dataclass: its fields are its
configuration, and it cannot store anything about a session. One selector serves every session
of an engine, so a value it stored for one session would be read by the next.

In [ ]:
@dataclass(frozen=True)
class EarlyRandomesqueSelector(BaseSelector):
  """Draw among the most informative items while the estimate is rough, then give the most informative."""

  pool_size: int
  until_se: float

  @override
  def select(self, context: StepContext) -> int:
    candidates = remaining(context)
    information = context.item_bank.information(context.theta)[candidates]
    if context.se <= self.until_se:
      return pick_best(candidates, information, context.rng)
    return int(context.rng.choice(best_pool(candidates, information, self.pool_size)))


POOL_SIZE = 10
UNTIL_SE = 0.7
TEST_LENGTH = 15

selector = EarlyRandomesqueSelector(POOL_SIZE, UNTIL_SE)
engine = engine_with(selector, MaxLengthStopper(TEST_LENGTH))

## Call It on a Session

A strategy can be called directly, which is how to test one. `engine.context(session, context)`
returns the step context the selector would get for the session's next item; given the run
context, it carries the run's generator and exposure rates. Here a simulated
examinee of ability 1 answers two items, and the selector is asked 50 times what to give next:

In [ ]:
context = RunContext(rng=numpy.random.default_rng(RUN_SEED))
session = engine.start_session("demo", context, true_theta=1.0)
for _ in range(2):
  engine.step(session, SimulatedResponseProvider(), context)

view = engine.context(session, context)
print(view.administered_items, view.responses, view.excluded_items, view.exposure_rates)
print(session.estimates[-1])
print(sorted({selector.select(view) for _ in range(50)}), session.administered_count)

The context holds the two items given and their responses, no excluded items, and no exposure
rates, since this run context has no exposure tracker. Its `theta` and `se` are those of the
session's newest `Estimate`: the ability estimate, its standard error, and `evaluations`, how many
times the estimator evaluated its objective, here EAP's posterior at its 121 grid points. The
standard error is above 0.7, so the selector draws: 50 calls give only 10 different items, the pool of
the 10 most informative. Calling a strategy changes nothing in the session, which still has two items.

## Run It in a Simulation

The selector goes wherever a built-in one does. Here it runs tests of `TEST_LENGTH` (15) items,
against `MaxInfoSelector`, with the same seed:

In [ ]:
max_info = SimulationRunner(engine_with(MaxInfoSelector(), MaxLengthStopper(TEST_LENGTH))).run(
  true_thetas, rng=RUN_SEED
)
early = SimulationRunner(engine).run(true_thetas, rng=RUN_SEED)
for name, result in (("max-info", max_info), ("early randomesque", early)):
  first_items = {items[0] for items in result.administered_items}
  print(
    f"{name}: {len(first_items)} different first items, "
    f"highest exposure rate {numpy.max(result.exposure_rates):.2f}, RMSE {result.rmse:.3f}"
  )
drawn = [sum(estimate.se > UNTIL_SE for estimate in session.estimates[:-1]) for session in early.sessions]
print(f"items drawn at random per test: {numpy.mean(drawn):.1f}")

again = SimulationRunner(engine).run(true_thetas, rng=RUN_SEED)
print(again.administered_items == early.administered_items)

Maximum information gives all 200 examinees the same first item, at an exposure rate of 1.00.
The custom selector spreads the first items over 10, its pool, and the highest exposure rate falls
to 0.67. A session's items are selected with the estimates before its last, so counting those
above 0.7 gives the items drawn at random: 4.8 per test on average, about the first five. One
run's RMSEs do not rank the two selectors: compare them over replications, by their paired
differences, as the [comparing-strategies notebook](02_comparing_strategies.ipynb) does.

The second run with the same seed gives every examinee the same items: each random draw came
from the run's generator. A strategy that draws from anywhere else, such as `numpy.random`'s
global functions or a generator of its own, escapes the run's seed
([Reproducibility](https://douglasrizzo.com.br/catsim/reproducibility.html)).

## A Custom Stopper

`EstimateChangeStopper` ends the test once the newest response moves the ability estimate by less
than `max_change`: more items would change the result little. A stopper subclasses `BaseStopper`
and overrides `stop`, which receives the context with the estimate updated by the newest response
as `theta`.

The estimate before the newest response is not in the context, and the stopper must not keep it:
one stopper serves every session of an engine, and an application interleaves sessions, so a
stored estimate would belong to whichever session called last. The stopper recomputes it: it
gives its estimator the context without the newest item and response, and reads `theta` from the
`Estimate` it returns. That context still holds the newest estimate as `theta`. An estimator that
does not start from it, such as the EAP estimator used here, recomputes exactly the estimate the
session recorded; one that starts its search from it, such as `NumericalSearchEstimator`, can
give a different value. With one item, the estimate before it came from the initializer, which
the stopper cannot recompute, so it waits for a second item.

In [ ]:
@dataclass(frozen=True)
class EstimateChangeStopper(BaseStopper):
  """Stop once the newest response moves the ability estimate by less than `max_change`."""

  estimator: BaseEstimator
  max_change: float

  @override
  def stop(self, context: StepContext) -> bool:
    if len(context.administered_items) <= 1:
      return False
    before = dataclasses.replace(
      context, administered_items=context.administered_items[:-1], responses=context.responses[:-1]
    )
    return abs(context.theta - self.estimator.estimate(before).theta) < self.max_change


MAX_CHANGE = 0.05
MAX_ITEMS = 30

change = EstimateChangeStopper(estimator, MAX_CHANGE)
rule = StopRule(change, MaxLengthStopper(MAX_ITEMS))
change_engine = engine_with(MaxInfoSelector(), rule)

context = RunContext(rng=numpy.random.default_rng(RUN_SEED))
live = change_engine.start_session("live", context, true_theta=1.0)
change_engine.run_session(live, SimulatedResponseProvider(), context)
print(live.administered_count, [round(theta, 3) for theta in live.theta_history[-3:]])
print(rule.satisfied(change_engine.context(live)))

A `StopRule` combines the custom stopper with a cap of `MAX_ITEMS` (30), so that every test ends
even if the estimate keeps moving. The live session ends after 20 items: its last response moves
the estimate from 1.162 to 1.207, by less than `MAX_CHANGE` (0.05). Asked afterwards, the rule
names the custom stopper as the criterion that holds. It can answer after the fact because the stopper is pure: the same
context always gives the same answer.

In a simulation:

In [ ]:
result = SimulationRunner(change_engine).run(true_thetas, rng=RUN_SEED)
lengths = result.test_lengths
print(f"test lengths from {numpy.min(lengths)} to {numpy.max(lengths)}, mean {numpy.mean(lengths):.1f}")
held = numpy.array([change.stop(change_engine.context(session)) for session in result.sessions])
print(numpy.count_nonzero(held), numpy.unique(lengths[~held]))
print(f"final standard errors from {numpy.min(result.final_ses):.2f} to {numpy.max(result.final_ses):.2f}")

Test lengths range from 13 to 30 items, 21.5 on average. Called on each finished session, the
stopper holds for 193 of the 200; the other 7 all have 30 items: they reached the cap with the
estimate still moving. A small change of the estimate is not a small standard error: the final standard errors
range from 0.33 to 0.51. To end tests at a target precision, combine the custom stopper with a
`StandardErrorStopper` in the rule.

An estimator, an initializer or an exposure control is written the same way, from its base class;
[Extending catsim](https://douglasrizzo.com.br/catsim/extending.html) lists what each receives and returns. A custom strategy is
compared with the built-in ones over replications, as the
[comparing-strategies notebook](02_comparing_strategies.ipynb) does.